Environment

In [1]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [2]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [3]:
import json
import pycomet
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import NodePartitionFunction, EdgeBackward, LinearMEGraphNet

Initializing thermophysical models

In [4]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [5]:
# Load data
sol = h5todict(h5file='./../physics/solution.hdf5')
# Extract primitive variables
t, T = [sol[k] for k in ('t', 'T')]
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]
n = np.hstack([na, nm])
# Update thermochemistry
kin._update(n, T)

Construct graph

In [6]:
# Dimensions
nb_pts, nb_nodes = nm.shape

In [7]:
# Adjacency matrix (forward)
kf = 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
k_idx = np.where(kf > 0)

In [8]:
# Nodes
node_nm = nm[0].reshape(1,-1,1)
node_nm = np.tile(node_nm, (nb_pts,1,1))
node_Tt = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))
node_Ti = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))

In [9]:
# Edges
edge_kf = kf[k_idx].reshape(1,-1,1)
edge_kf = np.tile(edge_kf, (nb_pts,1,1))
edge_na = na[0].reshape(1,1,1)
edge_na = np.tile(edge_na, (nb_pts,1,1))

In [10]:
# Graph
graph = [node_nm, node_Tt, node_Ti, edge_kf, edge_na]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)
# Target
target = tf.convert_to_tensor(nm[...,None]) / const.UNA

Load linear master equation graph model

In [11]:
lays_args = {

    "node_pf": {
        "lay_id": "NodePartitionFunction",
        "kwargs": {
            "node_index_temp": {
                "int": 2,
                "chem": 1,
                "trans": 1
            },
            "specie_thermos": {
                "filename": "./data/sp_th.json",
                "thermo_process": 0,
                "eos_model": "PerfectGas"
            }
        }
    },

    "edge_bwd": {
        "lay_id": "EdgeBackward",
        "kwargs": {
            "node_attr_index_pf": 3,
            "static_fwd_edge_index": "./data/fwd_edge_index.txt"
        }
    },

    "lin_me_gnet": {
        "lay_id": "LinearMEGraphNet",
        "kwargs": {
            "node_attr_indices": [0]
        }
    }

}

In [12]:
tf.keras.backend.clear_session()

lays = {
    "node_pf": NodePartitionFunction(**lays_args["node_pf"]["kwargs"]),
    "edge_bwd": EdgeBackward(**lays_args["edge_bwd"]["kwargs"]),
    "lin_me_gnet": LinearMEGraphNet(**lays_args["lin_me_gnet"]["kwargs"])
}

def get_graph(inputs):
    node_nm, node_Tt, node_Ti, edge_kf, edge_na = inputs
    node_attr = tf.concat([node_nm,node_Tt,node_Ti], axis=-1)
    edge_attr = edge_kf*edge_na
    return (node_attr, edge_attr)

def call(inputs, training=False):
    graph = get_graph(inputs)
    # Node partition functions
    graph = lays["node_pf"](graph, training=training)
    # Edge backward
    graph = lays["edge_bwd"](graph, training=training)
    # Graph processor
    y = lays["lin_me_gnet"](graph, training=training) + inputs[0]
    return y / const.UNA

In [13]:
shapes = tf.nest.map_structure(lambda x: tuple(x.shape[1:]), graph)
inputs = [tf.keras.Input(shape) for shape in shapes]
model = tf.keras.Model(inputs, call(inputs))

In [14]:
def add_noise(x, stddev=0.02):
    rnd = tf.random.get_global_generator()
    dx = rnd.normal(
        shape=tf.shape(x),
        mean=0.0,
        stddev=stddev,
        dtype=x.dtype,
    )
    return x * (1 + dx)

Compute Jacobians

In [15]:
graphn = tf.nest.map_structure(tf.identity, graph)
# graphn = tf.nest.map_structure(lambda x: add_noise(x, stddev=0.2), graphn)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  g.watch(graphn[3])
  y = model(graphn)
dy_dTi = tf.squeeze(g.batch_jacobian(y, graphn[2]))
dy_dkf = tf.squeeze(g.batch_jacobian(y, graphn[3]))

In [16]:
np.savetxt("./dy_dTi.txt", dy_dTi[-1].numpy(), fmt='%.8e')
np.savetxt("./dy_dkf.txt", dy_dkf[-1].numpy(), fmt='%.8e')

Compute gradients

In [18]:
def loss(y_true, y_pred, axis=0):
    # err = tf.math.squared_difference(y_pred, y_true)
    err = tf.abs(y_pred, y_true) / y_true
    err = tf.reduce_mean(err, axis=axis)
    return tf.reduce_sum(err)

In [19]:
graphn = tf.nest.map_structure(tf.identity, graph)
# graphn = tf.nest.map_structure(lambda x: add_noise(x, stddev=0.2), graphn)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  g.watch(graphn[3])
  y = model(graphn)
  e = loss(target, y, axis=0)
de_dTi = tf.squeeze(g.gradient(e, graphn[2]))
de_dkf = tf.squeeze(g.gradient(e, graphn[3]))